# 8.02 从零实现 RLT：冻结 VLA，用小网络学"最后一厘米"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/08_vla_rl/8.02_rlt.ipynb)


## 为什么要学

RLT（RL Token，Physical Intelligence 2025）是你正在 ALOHA 上用于线头对插的方法。它的核心想法：

> VLA 已经"大致会做"了，缺的是最后那一点精度。那就**冻结 VLA**，从它身上"抽出"一个紧凑的表示（RL token），再训练一个**很小的 actor-critic**，在 VLA 给出的参考动作附近做修正。

本课在 mini-π0 上完整实现 RLT 的两个阶段，并对照 RLinf 的实现（`rlt_token_transformer.py`、`rlt_mlp_policy.py`、`fsdp_rlt_ac_policy_worker.py`）。

---

## 1. 任务：一个"插不到底"的示教者

我们用有**系统性偏差**的示教训练基础 VLA：示教者总是停在离方块中心 5cm 的地方（沿着它过来的方向），就像插线时总是差一点没插到底。评估时要求夹爪到达中心 3cm 以内。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.envs import ToyTableEnv, CharTokenizer
from vla_ladder.pipelines import get_base_vla, build_vla_dataset, obs_to_tensors
from vla_ladder.models.rlt import RLTLayer, RLTokenTransformer, RLTActorCritic, mlp
from vla_ladder.utils import set_seed, get_device, count_params, moving_average

device = get_device()
tok = CharTokenizer()
OFFSET, RADIUS, C = 0.05, 0.03, 5          # 示教偏差、成功半径、RL 每一步执行的动作数（动作块的前 C 步）

# 第一次运行会训练一个基础 VLA（VLM 预训练 + 1500 条有偏示教，GPU 约 3 分钟），之后从缓存加载
vla = get_base_vla(device=device, expert_offset=OFFSET, demo_steps=30, n_episodes=1500)
vla.eval()
for p in vla.parameters():
    p.requires_grad_(False)                # 冻结 VLA：本课从头到尾都不会修改它
N_IMG = vla.vision.n_patches

In [ ]:
@torch.no_grad()
def rollout(policy_fn, n=200, seed=123, radius=RADIUS):
    """policy_fn(obs) -> 动作块 (n, C, 2)。返回 (成功率, 最终位置与目标的距离, 环境)"""
    e = ToyTableEnv(num_envs=n, seed=seed, success_radius=radius)
    obs = e.reset()
    while not e.done.all():
        a = policy_fn(obs).cpu().numpy()
        for k in range(C):
            obs, _, _, info = e.step(a[:, k])
    return info["success"].mean(), info["dist"], e

def vla_policy(obs):
    return vla.sample_actions(*obs_to_tensors(obs, tok, device)).clamp(-1, 1)[:, :C]

sr_base, dist_base, _ = rollout(vla_policy)
print(f"基础 VLA：成功率 {sr_base:.2f}，最终距离中位数 {np.median(dist_base):.3f}（示教偏差 {OFFSET}）")

基础 VLA 学会了示教者的坏习惯：停在偏差点附近。

---

## 2. Stage 1：RL token，把 VLA 的"理解"压缩成一个向量

actor-critic 需要知道"现在的场景是什么样的"：目标在哪、夹爪在哪。VLA 的前缀输出（VLM 最后一层，图像 token × 36，每个 128 维）包含了这些信息，但太大了（RLinf 中是 768 × 2048）。直接喂给 MLP 不现实。

RLT 的做法是训练一个小 Transformer，把前缀**压缩成 1 个 token**：

```text
编码器： [p_1, p_2, ..., p_L, <RL>] ──自注意力──> 取最后一个位置 = z_rl
解码器： [z_rl, p_1, ..., p_{L-1}] ──因果自注意力──> 重建 [p_1, ..., p_L]（teacher forcing）
损失：   MSE(重建, 原前缀)；前缀 detach（VLA 冻结）
```

**为什么用重建损失？** 它是一个信息瓶颈：解码器只能通过 z_rl 了解整个前缀（还有自己已经看过的前面几个 token），所以 z_rl 必须包含前缀中最重要的信息。它不需要任何奖励或动作标签，用 VLA 自己的数据就能训练。

> RLinf 的配置 `rlt_image_only=True`：只用**图像 token** 的前缀输出。因为前缀内部是双向注意力（7.01 课），VLM 最后一层的图像 token 已经"读过"了指令，所以其中也包含"要去哪个方块"的信息。

### 练习 1：RL token 编码器

参考实现 `vla_ladder.models.rlt.RLTokenTransformer` 已经写好了解码器和损失，请实现编码器 `encode`。

In [ ]:
class MyRLToken(RLTokenTransformer):
    def encode(self, prefix, mask=None):
        """prefix: (B, L, input_dim) -> z_rl (B, embed_dim)"""
        B, L, _ = prefix.shape
        # TODO:
        # 1) x = cat([in_proj(prefix), rl_token 扩展成 (B, 1, D)], dim=1) + enc_pos[:L+1]
        # 2) 依次过 self.encoder 中的每一层（本课不用 mask）
        # 3) 返回最后一个位置 x[:, -1]
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MyRLToken(RLTokenTransformer):
    def encode(self, prefix, mask=None):
        """prefix: (B, L, input_dim) -> z_rl (B, embed_dim)"""
        B, L, _ = prefix.shape
        x = torch.cat([self.in_proj(prefix), self.rl_token.expand(B, -1, -1)], 1) + self.enc_pos[: L + 1]
        for layer in self.encoder:
            x = layer(x)
        return x[:, -1]

In [ ]:
@torch.no_grad()
def vla_prefix(images, lang, lang_mask, state):
    """VLA 前缀最后一层的输出中的图像 token：(B, 36, 128)"""
    _, _, prefix_out = vla.prefix_cache(images, lang, lang_mask, state)
    return prefix_out[:, :N_IMG]

data = build_vla_dataset(300, 10, device=device, expert_offset=OFFSET, demo_steps=30, demo_radius=0.0)   # 用 VLA 自己的示教数据即可
set_seed(0)
rlt = MyRLToken(input_dim=vla.cfg.vlm_width, embed_dim=64, seq_len=N_IMG).to(device)
opt = torch.optim.AdamW(rlt.parameters(), lr=1e-3)
t0 = time.time()
for step in range(3000):
    i = torch.randint(0, len(data["state"]), (128,), device=device)
    pre = vla_prefix(data["image"][i].float() / 255, data["lang"][i], data["lang_mask"][i], data["state"][i])
    loss = rlt.loss(pre)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0 or step == 2999:
        print(f"  [RL token] step {step:4d}  重建误差 {loss.item():.4f}  ({time.time() - t0:.0f}s)")
rlt.eval()
for p in rlt.parameters():
    p.requires_grad_(False)                # Stage 2 中 RL token 编码器也冻结
assert loss.item() < 0.1
print("通过 ✅")

---

## 3. Stage 2：在参考动作附近学修正的 actor-critic

### MDP 的定义（以"动作块"为单位）

| | 定义 |
|---|---|
| 状态 | $z_{rl}$（RL token）+ 本体状态（夹爪位置） |
| 参考动作 $a_{ref}$ | VLA 给出的动作块的前 $C=5$ 步（展平成 10 维） |
| 动作 $a$ | actor 输出的 5 步动作块 |
| 奖励 | 这 5 步中首次到达目标：1，否则 0（**稀疏奖励**） |
| 终止 | 成功即终止 |

### actor：残差 + 固定标准差 + 参考动作 dropout

对照 RLinf 的 `RLTMLPPolicy`：

$$
\pi(a\mid z,s,a_{ref})=\mathcal N\big(\underbrace{a_{ref}+\Delta_\theta(z,s,\tilde a_{ref})}_{\text{均值}},\ \sigma^2I\big)
$$

- **残差输出**（`actor_output="residual"`）：$\Delta$ 头**零初始化**，所以一开始 actor 完全等于 VLA（1.08 课的技巧）；
- **固定的小 $\sigma$**（`fixed_std`）：只在参考动作附近小范围探索。真机上大范围随机探索既危险又低效；
- **参考动作 dropout**（`reference_dropout_prob`）：训练时以一定概率把**输入**中的 $\tilde a_{ref}$ 置零，迫使 actor 也去看 $z$ 和本体状态，而不是只会"抄"参考动作。注意输出的残差基准 $a_{ref}$ 不丢弃。

### 练习 2：actor 的均值

In [ ]:
class MyRLTActorCritic(RLTActorCritic):
    def actor_mean(self, z, proprio, a_ref, ref_dropout=0.0):
        """返回 a_ref + Δ(z, proprio, ref_in)，其中 ref_in 在训练时按 ref_dropout 的概率（逐样本）置零"""
        # TODO: 训练模式且 ref_dropout > 0 时：keep = (rand(B, 1) > ref_dropout)；ref_in = a_ref * keep
        #       return a_ref + self.actor(cat([z, proprio, ref_in], -1))
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class MyRLTActorCritic(RLTActorCritic):
    def actor_mean(self, z, proprio, a_ref, ref_dropout=0.0):
        """返回 a_ref + Δ(z, proprio, ref_in)，其中 ref_in 在训练时按 ref_dropout 的概率（逐样本）置零"""
        ref_in = a_ref
        if ref_dropout > 0 and self.training:
            keep = (torch.rand(a_ref.shape[0], 1, device=a_ref.device) > ref_dropout).float()
            ref_in = a_ref * keep
        return a_ref + self.actor(torch.cat([z, proprio, ref_in], -1))

In [ ]:
# ✅ 自测：零初始化的残差 actor 一开始就等于参考动作
ac_test = MyRLTActorCritic(64, 2, 2 * C).to(device)
z, s, ref = torch.randn(8, 64, device=device), torch.randn(8, 2, device=device), torch.randn(8, 2 * C, device=device)
assert torch.allclose(ac_test.actor_mean(z, s, ref, ref_dropout=0.5), ref)
print("通过 ✅")

### 练习 3：critic 与 actor 的损失

$$
y=r+\gamma(1-d)\min_{j}Q_{\bar\phi_j}(z',s',a'),\quad a'\sim\pi_{\bar\theta}(\cdot\mid z',s',a'_{ref})
$$

$$
L_Q=\sum_j\big(Q_{\phi_j}(z,s,a)-y\big)^2,\qquad
L_\pi=-\min_jQ_{\phi_j}\big(z,s,\mu_\theta\big)+\beta\,\|\mu_\theta-a_{ref}\|^2
$$

和 RLinf 的 `actor_loss = -q_weight * qf_pi.mean() + bc_weight * bc_loss` 一致。

In [ ]:
def my_rlt_losses(ac, ac_targ, b, gamma=0.95, bc_weight=0.01, ref_dropout=0.2):
    """b: 字典，含 z, proprio, a_ref, action, reward, next_z, next_proprio, next_a_ref, done。
    返回 (critic_loss, actor_loss_fn)；actor_loss_fn() 返回 (actor_loss, bc)，要在 critic 更新之后再调用"""
    with torch.no_grad():
        # TODO: a2 = ac_targ.act(next_z, next_proprio, next_a_ref)；y = reward + γ(1-done)·min(ac_targ.q(...))
        raise NotImplementedError
    q1, q2 = ac.q(b["z"], b["proprio"], b["action"])
    critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)

    def actor_loss_fn():
        # TODO: mean = ac.actor_mean(z, proprio, a_ref, ref_dropout)；q = min(ac.q(z, proprio, mean))
        #       bc = ((mean - a_ref)²).sum(-1).mean()；return -q.mean() + bc_weight * bc, bc
        raise NotImplementedError

    return critic_loss, actor_loss_fn

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def my_rlt_losses(ac, ac_targ, b, gamma=0.95, bc_weight=0.01, ref_dropout=0.2):
    """b: 字典，含 z, proprio, a_ref, action, reward, next_z, next_proprio, next_a_ref, done。
    返回 (critic_loss, actor_loss_fn)；actor_loss_fn() 返回 (actor_loss, bc)，要在 critic 更新之后再调用"""
    with torch.no_grad():
        a2 = ac_targ.act(b["next_z"], b["next_proprio"], b["next_a_ref"])
        q1t, q2t = ac_targ.q(b["next_z"], b["next_proprio"], a2)
        y = b["reward"] + gamma * (1 - b["done"]) * torch.min(q1t, q2t)
    q1, q2 = ac.q(b["z"], b["proprio"], b["action"])
    critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)

    def actor_loss_fn():
        mean = ac.actor_mean(b["z"], b["proprio"], b["a_ref"], ref_dropout)
        q1p, q2p = ac.q(b["z"], b["proprio"], mean)
        bc = (mean - b["a_ref"]).pow(2).sum(-1).mean()
        return -torch.min(q1p, q2p).mean() + bc_weight * bc, bc

    return critic_loss, actor_loss_fn

---

## 4. 在线训练（GPU 约 2~3 分钟）

In [ ]:
@torch.no_grad()
def observe(obs):
    """一次 VLA 前向：得到 RL token、本体状态、参考动作块"""
    I, L, M, S = obs_to_tensors(obs, tok, device)
    _, _, prefix_out = vla.prefix_cache(I, L, M, S)
    z = rlt.encode(prefix_out[:, :N_IMG])
    a_ref = vla.sample_actions(I, L, M, S).clamp(-1, 1)[:, :C].reshape(len(S), -1)
    return z, S, a_ref

class Replay:
    def __init__(self, cap=200_000):
        self.cap, self.ptr, self.size, self.data = cap, 0, 0, {}

    def add(self, tr):
        n = next(iter(tr.values())).shape[0]
        idx = (torch.arange(n, device=device) + self.ptr) % self.cap
        for k, v in tr.items():
            if k not in self.data:
                self.data[k] = torch.zeros((self.cap,) + tuple(v.shape[1:]), device=device)
            self.data[k][idx] = v.float()
        self.ptr, self.size = (self.ptr + n) % self.cap, min(self.size + n, self.cap)

    def sample(self, B):
        i = torch.randint(0, self.size, (B,), device=device)
        return {k: v[i] for k, v in self.data.items()}

def rlt_policy(ac, deterministic=True):
    def fn(obs):
        z, S, ref = observe(obs)
        return ac.act(z, S, ref, deterministic=deterministic).clamp(-1, 1).view(len(S), C, 2)
    return fn

set_seed(0)
ac = MyRLTActorCritic(z_dim=64, proprio_dim=2, chunk_dim=2 * C, fixed_std=0.15).to(device)
ac_targ = copy.deepcopy(ac)
opt_q = torch.optim.Adam(list(ac.q1.parameters()) + list(ac.q2.parameters()), lr=3e-4)
opt_pi = torch.optim.Adam(ac.actor.parameters(), lr=3e-4)
buf, N_ENV, log = Replay(), 64, {"iter": [], "train": [], "eval": []}
t0 = time.time()
for it in range(100):
    env = ToyTableEnv(num_envs=N_ENV, seed=1000 + it, success_radius=RADIUS)
    obs = env.reset(); z, S, ref = observe(obs)
    while not env.done.all():
        alive = torch.tensor(~env.done, device=device)
        with torch.no_grad():
            a = ac.act(z, S, ref).clamp(-1, 1)                        # 带固定 σ 的探索
        a_np = a.view(N_ENV, C, 2).cpu().numpy(); prev_succ = env.success.copy()
        for k in range(C):
            obs, _, _, info = env.step(a_np[:, k])
        reward = torch.tensor(info["success"] & ~prev_succ, device=device).float()
        z2, S2, ref2 = observe(obs)
        tr = dict(z=z, proprio=S, a_ref=ref, action=a, reward=reward, next_z=z2, next_proprio=S2, next_a_ref=ref2,
                  done=torch.tensor(info["success"], device=device).float())
        buf.add({k: v[alive] for k, v in tr.items()})                # 只存还没结束的环境的转移
        z, S, ref = z2, S2, ref2
        if buf.size > 2000:
            for _ in range(8):                                         # 每采一步更新 8 次（高 UTD，样本效率高）
                critic_loss, actor_loss_fn = my_rlt_losses(ac, ac_targ, buf.sample(256))
                opt_q.zero_grad(); critic_loss.backward(); opt_q.step()
                actor_loss, bc = actor_loss_fn()
                opt_pi.zero_grad(); actor_loss.backward(); opt_pi.step()
                with torch.no_grad():                                  # 目标网络 Polyak 更新
                    for pt, p in zip(ac_targ.parameters(), ac.parameters()):
                        pt.mul_(0.995).add_(p, alpha=0.005)
    log["iter"].append(it); log["train"].append(info["success"].mean())
    if it % 20 == 19:
        ac.eval(); sr, _, _ = rollout(rlt_policy(ac)); ac.train()
        log["eval"].append((it, sr))
        print(f"iter {it:3d}  训练（带探索）成功率 {np.mean(log['train'][-20:]):.2f}  评估（确定性）{sr:.2f}  buffer {buf.size}  ({time.time() - t0:.0f}s)")
ac.eval()

In [ ]:
sr_rlt, dist_rlt, e_rlt = rollout(rlt_policy(ac))
print(f"基础 VLA {sr_base:.2f}  ->  RLT {sr_rlt:.2f}")
assert sr_rlt > sr_base + 0.3
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(moving_average(log["train"], 5), label="train (with exploration)")
ax[0].plot(*zip(*log["eval"]), "o-", label="eval (deterministic)"); ax[0].axhline(sr_base, ls="--", c="gray", label="base VLA")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("success"); ax[0].legend()
ax[1].hist(dist_base, bins=40, alpha=0.6, label="base VLA"); ax[1].hist(dist_rlt, bins=40, alpha=0.6, label="RLT")
ax[1].axvline(RADIUS, c="k", ls="--"); ax[1].set_xlabel("final distance to target"); ax[1].legend()
plt.tight_layout(); plt.show()
print("通过 ✅ RLT 学会了'插到底'")

---

## 5. 关键超参数与 RLinf 配置的对应

| 本课 | RLinf 配置 / 代码 | 作用 |
|---|---|---|
| `fixed_std=0.15` | `fixed_std` / `fixed_gripper_std` | 探索范围：只在参考动作附近小范围试探 |
| `bc_weight=0.01` | `bc_weight`（可分 warmup / online 两段） | 离参考动作多远 |
| 残差 + 零初始化 | `actor_output: residual` | 起点 = VLA，训练一开始不会变差 |
| `ref_dropout=0.2` | `reference_dropout_prob` | 防止 actor 只会抄参考动作 |
| twin Q 取 min | `q_head_type`、`_min_twin_q` | 抑制 Q 高估 |
| 每采一步更新 8 次 | UTD（update-to-data）比 | 样本效率：真机上每条数据都很贵 |

**BC 权重的影响**（我们在本任务上做过的扫描，成功半径 0.03，确定性评估）：$\beta=0.1$ 时约 0.65~0.75，$\beta=0.01$ 时约 0.9。BC 正则太强，actor 就不敢离开"插不到底"的参考动作。但在真机上，β 太小又可能让 actor 学出 VLA 没见过的危险动作。**这是 RLT 最需要根据任务调的超参数**，RLinf 允许在 warmup 阶段用较大的 β、在线阶段逐步减小。

> **探索噪声与评估**：训练时带探索噪声的策略成功率往往**高于**确定性策略。原因是稀疏奖励只奖励"进入半径"，带噪声的"微小抖动"能帮它碰巧进入。真机插接时，人类也会轻微晃动来"找"插口。RLinf 的 `execute_actor_mean` 控制真机执行时用均值还是采样。

### 真机 RLT 还多了什么？

1. **人工干预（接管）数据**：操作员在机器人要失败时接管（3.01 课的 HG-DAgger）。这些数据同样进入 replay buffer，BC 正则的目标对它们换成**人的动作**而不是 VLA 的参考动作（RLinf 中 `bc_target = where(human_mask, action_chunk, ref_chunk)`）。
2. **奖励来源**：成功检测器或操作员按键标注。
3. **只在"关键阶段"启用 RL**：接近阶段交给 VLA，只在插入阶段让 actor 接管（RLinf 中的 `route` 逻辑）。

### 课后实验

1. 把 `bc_weight` 改成 1.0 重跑，观察确定性成功率。
2. 把 `fixed_std` 改成 0.05 或 0.5，观察训练曲线：探索太小或太大分别会怎样？
3. 去掉残差（让 actor 直接输出动作块，最后一层不零初始化），训练初期的成功率会怎样变化？
4. 把 `rlt.encode(...)` 换成"对前缀做平均池化"，RL token 的作用有多大？

---

## 小结

- RLT 两阶段：(1) 用重建损失把 VLA 前缀压缩成 RL token；(2) 冻结 VLA 和 RL token，在线训练小 actor-critic。
- actor 以参考动作为输入和残差基准、零初始化、固定小 σ、参考动作 dropout；actor 损失 = $-Q$ + BC 正则。
- 全程不修改 VLA，就把成功率从示教水平提升到远超示教。